In [2]:
import os
import glob
from datetime import datetime
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

# -------------------------------------------------------------
# Configuration & Environment Variables
# -------------------------------------------------------------
mdb_path = os.getenv("MDB")
silver_u_dir = os.getenv("SILVERU")

if not mdb_path or not os.path.exists(mdb_path):
    raise FileNotFoundError(f"❌ MDB environment file path not found or invalid: {mdb_path}")

if not silver_u_dir or not os.path.exists(silver_u_dir):
    raise FileNotFoundError(f"❌ SILVERU environment directory not found or invalid: {silver_u_dir}")

# -------------------------------------------------------------
# Load & Aggregate SILVERU Update Files
# -------------------------------------------------------------
excel_files = glob.glob(os.path.join(silver_u_dir, "*.xlsx"))
if not excel_files:
    print(f"⚠️ No Excel update files found in SILVERU directory: {silver_u_dir}")
    exit()

print(f"📂 Found {len(excel_files)} file(s) in SILVERU folder.")

silver_dfs = []
required_silver_cols = {"Proposal No.", "Original Status", "Portal Status", "Flag"}

for file in excel_files:
    df_temp = pd.read_excel(file)
    if required_silver_cols.issubset(df_temp.columns):
        silver_dfs.append(df_temp)

if not silver_dfs:
    raise ValueError("❌ None of the Excel files in SILVERU contain the required columns.")

# Combine all SILVERU files
df_updates = pd.concat(silver_dfs, ignore_index=True)

# Clean and normalize strings
df_updates["Proposal No."] = df_updates["Proposal No."].astype(str).str.strip()
df_updates["Flag"] = df_updates["Flag"].astype(str).str.strip().str.lower()
df_updates["Portal Status"] = df_updates["Portal Status"].fillna("").astype(str).str.strip()

# Filter for records with Flag == 'change' or 'eliminated'
df_updates = df_updates[df_updates["Flag"].isin(["change", "eliminated"])]

# Deduplicate by Proposal No. (keeping the latest occurrence)
df_updates = df_updates.drop_duplicates(subset=["Proposal No."], keep="last")

print(f"🔍 Total records marked for update ('change' / 'eliminated'): {len(df_updates)}")

# -------------------------------------------------------------
# Read and Update MDB File
# -------------------------------------------------------------
print(f"📂 Reading MDB master file: {mdb_path}")
df_mdb = pd.read_excel(mdb_path)

if "Proposal No." not in df_mdb.columns or "Proposal Status" not in df_mdb.columns:
    raise KeyError("❌ MDB file must contain 'Proposal No.' and 'Proposal Status' columns.")

# Ensure 'Comment' column exists
if "Comment" not in df_mdb.columns:
    df_mdb["Comment"] = ""

df_mdb["Proposal No."] = df_mdb["Proposal No."].astype(str).str.strip()

# Create status lookup map
status_map = dict(zip(df_updates["Proposal No."], df_updates["Portal Status"]))

cdate = datetime.now().strftime("%Y%m%d")
comment_stamp = f"Updated {cdate}"

# Update matching rows in MDB dataframe
updated_count = 0
for idx, row in df_mdb.iterrows():
    proposal_no = row["Proposal No."]
    if proposal_no in status_map:
        df_mdb.at[idx, "Proposal Status"] = status_map[proposal_no]
        df_mdb.at[idx, "Comment"] = comment_stamp
        updated_count += 1

print(f"✨ Successfully updated {updated_count} matching proposals in MDB file.")

# -------------------------------------------------------------
# Sort and Save MDB File
# -------------------------------------------------------------
# Sort MDB file based on 'Comment' column
df_mdb = df_mdb.sort_values(by="Comment", ascending=True, na_position="last").reset_index(drop=True)

df_mdb.to_excel(mdb_path, index=False)
print(f"🎉 Updated file saved and sorted at: {mdb_path}")

# -------------------------------------------------------------
# Cleanup: Delete Update Files from SILVERU
# -------------------------------------------------------------
print(f"🧹 Cleaning up processed files in SILVERU directory...")
for file_path in excel_files:
    try:
        os.remove(file_path)
        print(f"  🗑️ Deleted: {os.path.basename(file_path)}")
    except Exception as e:
        print(f"  ⚠️ Could not delete {os.path.basename(file_path)}: {e}")

print("✅ Cleanup complete.")

📂 Found 1 file(s) in SILVERU folder.
🔍 Total records marked for update ('change' / 'eliminated'): 2483
📂 Reading MDB master file: F:\Chimney Work\Marketing\LeadGen\Data Architecture\2 - Silver\MasterDB.xlsx
✨ Successfully updated 2483 matching proposals in MDB file.
🎉 Updated file saved and sorted at: F:\Chimney Work\Marketing\LeadGen\Data Architecture\2 - Silver\MasterDB.xlsx
🧹 Cleaning up processed files in SILVERU directory...
  🗑️ Deleted: NonMin_Updated_Status_20260806.xlsx
✅ Cleanup complete.
